Local-only CGM compute: produce a gaussian-line phase payload for the
dedicated Windows hardware runner.

Parallel to ``scripts/testfile_lg.py`` but targeting a 1D line with
Gaussian cross-section via ``SLM.gaussian_line_target``.  Shared CGM +
optics config with the LG01 baseline so the per-shape diff on the
camera is attributable to the target choice alone.

Next step after running this script::

    ./push_run.sh payload/gline/testfile_gline_payload.npz


In [ ]:
from __future__ import annotations

import json
import os
import sys
import time

import numpy as np
import torch
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt

from slm.cgm import CGM_phase_generate, CGMConfig, _initial_phase
from slm.generation import SLM_class
from slm import imgpy
from slm.targets import mask_from_target


OUTPUT_DIR = "payload/gline"
PAYLOAD_PATH = f"{OUTPUT_DIR}/testfile_gline_payload.npz"
PARAMS_PATH = f"{OUTPUT_DIR}/testfile_gline_params.json"
PREVIEW_PATH = f"{OUTPUT_DIR}/testfile_gline_preview.pdf"


In [ ]:
# --- Capture parameters (used later by the Windows runner) ---
etime_us = 4000        # 4 ms exposure (kept low to avoid zero-order saturation on camera)
n_avg = 10             # average 10 frames
LUT = 207
fresnel_sd = 1000      # um -- compensates camera-focal-plane offset


In [ ]:
# --- Gaussian-line target parameters (1024^2 grid, focal pitch 15.83 um/px) ---
# 30 focal-px = 475 um length on focal plane = 258 camera-px.  sigma=2 = 17 camera-px.
gline_length = 30.0         # px along line (~475 um)
gline_width_sigma = 2.0     # px perpendicular (~32 um 1-sigma)
gline_angle = 0.0           # horizontal
gline_phase_gradient = 0.0  # no linear phase ramp


In [ ]:
# --- CGM analytical initial phase (issue #12 iteration #4: places target on-camera) ---
# Paper's Table I values (R=4.5e-3, D=-pi/2, theta=pi/4) on our 4096^2 grid shift
# the target by ~2865 um off zero-order, OFF the 5617x7444 um camera FOV.  Use the
# reduced shift that hardware iteration #4 in issue #12 confirmed to land on-camera.
cgm_R = 0          # no quadratic lensing (R=4.5e-3 caused crescent artifact per issue #12 #1)
cgm_D = -np.pi / 6     # smaller linear phase offset (~950 um shift, on-camera)
cgm_theta = - np.pi / 4        # horizontal offset only (keeps target within camera row extent)
cgm_steepness = 9
cgm_max_iterations = 2000


In [ ]:
# --- 1. SLM_class setup (reads hamamatsu_test_config.json) ---
# Override arraySizeBit from default [12,12] (=4096^2) to [10,10] (=1024^2) so the
# CGM compute grid matches the SLM native short dimension (1024 rows).  This avoids
# the lossy central 1024x1024 crop in phase_to_screen that previously dropped
# fidelity from 0.98 -> 0.22.  See root-cause investigation.
SLM = SLM_class()
SLM.arraySizeBit = [10, 10]  # 1024 x 1024 compute grid
SLM.image_init(
    initGaussianPhase_user_defined=np.zeros((1024, 1024)), Plot=False,
)
W, H = SLM.SLMRes  # (1272, 1024)
cx, cy = W // 2, H // 2

correct = lambda screen: imgpy.SLM_screen_Correct(  # noqa: E731
    screen, LUT=LUT, correctionImgPath="calibration/CAL_LSH0905549_1013nm.bmp"
)

print(f"Compute grid: ({SLM.ImgResY}, {SLM.ImgResX})  "
      f"focal pitch = {SLM.Focalpitchx:.3f} um/px")
print(f"SLM native:   ({H}, {W})")


In [ ]:
# --- 2. Generate gaussian-line target on the 1024x1024 computation grid ---
targetAmp = SLM.gaussian_line_target(
    length=gline_length,
    width_sigma=gline_width_sigma,
    angle=gline_angle,
    phase_gradient=gline_phase_gradient,
)
print(
    f"\n[TARGET] gaussian line: length={gline_length:.0f}px "
    f"width_sigma={gline_width_sigma:.0f}px angle={gline_angle:.2f} rad "
    f"dtype={targetAmp.dtype} shape={targetAmp.shape} "
    f"nonzero={np.count_nonzero(targetAmp)}"
)


In [ ]:
# Inline target visualization (amplitude + phase, full + auto-zoom).
amp = np.abs(targetAmp)
phase_t = np.angle(targetAmp)
ny, nx = amp.shape
x_um = (np.arange(nx) - (nx - 1) / 2.0) * SLM.Focalpitchx
y_um = (np.arange(ny) - (ny - 1) / 2.0) * SLM.Focalpitchy
extent_full = [x_um[0], x_um[-1], y_um[-1], y_um[0]]

peak = float(amp.max())
support = amp >= (peak * 0.05) if peak > 0 else np.zeros_like(amp, dtype=bool)
if not np.any(support):
    support = amp > 0
rows, cols = np.where(support)
if rows.size > 0:
    margin = 10
    r0 = max(0, rows.min() - margin)
    r1 = min(ny, rows.max() + margin + 1)
    c0 = max(0, cols.min() - margin)
    c1 = min(nx, cols.max() + margin + 1)
else:
    r0, r1, c0, c1 = 0, ny, 0, nx
extent_zoom = [x_um[c0], x_um[c1 - 1], y_um[r1 - 1], y_um[r0]]

plt.figure(figsize=(10, 8))
plt.subplot(2, 2, 1)
plt.imshow(amp, cmap="magma", extent=extent_full, aspect="auto")
plt.colorbar(label="|targetAmp|"); plt.title("target amplitude")
plt.xlabel("x (um)"); plt.ylabel("y (um)")
plt.subplot(2, 2, 2)
plt.imshow(phase_t, cmap="twilight", extent=extent_full, aspect="auto")
plt.colorbar(label="phase (rad)"); plt.title("target phase")
plt.xlabel("x (um)"); plt.ylabel("y (um)")
plt.subplot(2, 2, 3)
plt.imshow(amp[r0:r1, c0:c1], cmap="magma", extent=extent_zoom, aspect="auto")
plt.colorbar(label="|targetAmp|"); plt.title("amplitude (zoom)")
plt.xlabel("x (um)"); plt.ylabel("y (um)")
plt.subplot(2, 2, 4)
plt.imshow(phase_t[r0:r1, c0:c1], cmap="twilight", extent=extent_zoom, aspect="auto")
plt.colorbar(label="phase (rad)"); plt.title("phase (zoom)")
plt.xlabel("x (um)"); plt.ylabel("y (um)")
plt.tight_layout(); plt.show()


In [ ]:
# --- 3. Build the paper's analytical initial phase ---
init_phi = _initial_phase(
    (int(SLM.ImgResY), int(SLM.ImgResX)),
    CGMConfig(R=cgm_R, D=cgm_D, theta=cgm_theta),
)


In [ ]:
# --- 4. Run CGM on the compute grid via torch/CUDA ---
# eta_min=0.05 forces CGM to find a higher-efficiency solution.  Default (0) lets
# CGM converge to F~1 with eta~0.5%, where the D-grating-deflected zero-order
# overwhelms the target shape on hardware.  eta_min=0.05 gives ~10x more light in
# the target region at a small fidelity cost (F ~0.99 still).
cgm_device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\n[CGM] running {cgm_max_iterations} iterations on "
      f"{SLM.ImgResY}x{SLM.ImgResX} grid (device={cgm_device})...")
t0 = time.perf_counter()
SLM_Phase = CGM_phase_generate(
    torch.tensor(SLM.initGaussianAmp),
    torch.from_numpy(init_phi),
    torch.from_numpy(targetAmp),
    max_iterations=cgm_max_iterations,
    steepness=cgm_steepness,
    eta_min=0.1,
    Plot=False,
)
cgm_wall_time = time.perf_counter() - t0
per_iter_ms = cgm_wall_time / max(cgm_max_iterations, 1) * 1000
print(f"[CGM] done in {cgm_wall_time:.2f} s "
      f"({per_iter_ms:.1f} ms/iter)")

# Wrap phase to [-pi, pi] before cropping.
phase_np = SLM_Phase.cpu().clone().numpy()
phase_wrapped = np.angle(np.exp(1j * phase_np))
SLM_screen_raw = SLM.phase_to_screen(phase_wrapped)


In [ ]:
# --- 5. Post-hoc Fresnel lens ---
fresnel = SLM.fresnel_lens_phase_generate(fresnel_sd, cx, cy)[0]
SLM_screen_shift = (
    (SLM_screen_raw.astype(np.int32) + fresnel.astype(np.int32)) % 256
).astype(np.uint8)


In [ ]:
# --- 6. Calibration correction ---
SLM_screen_final = correct(SLM_screen_shift)
print(f"\n[SCREEN] ready-to-display uint8 {SLM_screen_final.shape} "
      f"range=[{SLM_screen_final.min()}, {SLM_screen_final.max()}]")


In [ ]:
# Inline view of the final uint8 SLM screen.
plt.figure(figsize=(8, 6))
plt.imshow(SLM_screen_final, cmap="gray", vmin=0, vmax=255)
plt.colorbar(label="uint8")
plt.title(f"SLM screen (Fresnel + calibration applied) {SLM_screen_final.shape}")
plt.tight_layout(); plt.show()


In [ ]:
# --- 7. Diagnostic metrics ---
from slm.metrics import fidelity as _fidelity, efficiency as _efficiency
from slm.propagation import fft_propagate
from slm.targets import measure_region as _measure_region

region_np = _measure_region(targetAmp.shape, targetAmp, margin=5)
E_out_1024 = fft_propagate(SLM.initGaussianAmp * np.exp(1j * phase_wrapped))
F = _fidelity(E_out_1024, targetAmp, region_np)
eta = _efficiency(E_out_1024, region_np)
print(f"[METRICS] fidelity F={F:.6f}  (1-F={1-F:.2e})  efficiency eta={eta*100:.2f}%")


In [ ]:
# --- 8. Save payload.npz ---
np.savez_compressed(
    PAYLOAD_PATH,
    slm_screen=SLM_screen_final,
)
payload_size_kb = int(np.ceil(
    __import__("os").path.getsize(PAYLOAD_PATH) / 1024
))
print(f"\n[SAVE] payload:  {PAYLOAD_PATH}  ({payload_size_kb} KB)")


In [ ]:
# --- 9. Save params.json ---
params = {
    "algorithm": "CGM",
    "payload": PAYLOAD_PATH,
    "compute_grid": [int(SLM.ImgResY), int(SLM.ImgResX)],
    "slm_native": [int(H), int(W)],
    "focal_pitch_x_um_per_px": round(float(SLM.Focalpitchx), 4),
    "focal_pitch_y_um_per_px": round(float(SLM.Focalpitchy), 4),
    "runner_defaults": {
        "etime_us": etime_us,
        "n_avg": n_avg,
        "monitor": 1,
    },
    "fresnel_applied_on_linux": True,
    "fresnel_shift_distance_um": fresnel_sd,
    "fresnel_center_xy_px": [int(cx), int(cy)],
    "calibration_applied_on_linux": True,
    "calibration_bmp": "calibration/CAL_LSH0905549_1013nm.bmp",
    "LUT": LUT,
    "target": "gaussian_line",
    "gline_length_px": gline_length,
    "gline_length_um_focal": round(float(gline_length * SLM.Focalpitchx), 2),
    "gline_width_sigma_px": gline_width_sigma,
    "gline_angle_rad": gline_angle,
    "gline_phase_gradient": gline_phase_gradient,
    "cgm_max_iterations": cgm_max_iterations,
    "cgm_steepness": cgm_steepness,
    "cgm_R": cgm_R,
    "cgm_D": cgm_D,
    "cgm_theta": cgm_theta,
    "cgm_wall_time_s": round(cgm_wall_time, 3),
    "cgm_per_iter_ms": round(per_iter_ms, 2),
    "cgm_device": cgm_device,
    "final_fidelity": round(float(F), 6),
    "final_one_minus_fidelity": float(f"{1 - F:.3e}"),
    "final_efficiency": round(float(eta), 6),
    "timestamp_local": time.strftime("%Y-%m-%dT%H:%M:%S", time.localtime()),
}
with open(PARAMS_PATH, "w", encoding="utf-8") as f:
    json.dump(params, f, ensure_ascii=False, indent=2)
print(f"[SAVE] params:  {PARAMS_PATH}")


In [ ]:
# --- 10. Save preview.pdf ---
# 6-panel inline diagnostic (replaces the legacy preview PDF).
from slm.targets import mask_from_target
target_mask = mask_from_target(targetAmp)
try:
    E_out_diag = E_out_4096
except NameError:
    E_out_diag = E_out_1024
try:
    slm_phase_full_diag = SLM_Phase.cpu().numpy()
except (NameError, AttributeError):
    slm_phase_full_diag = phase_wrapped

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes[0, 0].imshow(SLM.initGaussianAmp, cmap="viridis")
axes[0, 0].set_title("Input amplitude |S|")
axes[0, 1].imshow(np.abs(targetAmp) ** 2, cmap="hot")
axes[0, 1].set_title("Target |tau|^2 (Gaussian line)")
target_phase_masked = np.where(target_mask > 0, np.angle(targetAmp), np.nan)
axes[0, 2].imshow(target_phase_masked, cmap="twilight", vmin=-np.pi, vmax=np.pi)
axes[0, 2].set_title("Target phase arg(tau)")
out_int = (np.abs(E_out_diag) ** 2) * region_np
axes[1, 0].imshow(out_int, cmap="hot")
axes[1, 0].set_title(f"Output |E_out|^2\nF={F:.4f}, eta={100*eta:.2f}%")
out_phase_masked = np.where(target_mask > 0, np.angle(E_out_diag), np.nan)
axes[1, 1].imshow(out_phase_masked, cmap="twilight", vmin=-np.pi, vmax=np.pi)
axes[1, 1].set_title("Output phase arg(E_out)")
axes[1, 2].imshow(SLM_screen_final, cmap="gray", vmin=0, vmax=255)
axes[1, 2].set_title(f"SLM screen (Fresnel+calib applied)\n{SLM_screen_final.shape} uint8")
for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()


In [ ]:
# --- 11. Next-step hint ---
print()
print("=" * 72)
print("Payload ready.  Next step (pushes to Windows and runs the experiment):")
print()
print(f"    ./push_run.sh {PAYLOAD_PATH}")
print()
print("=" * 72)
